# CVAS Cloud Source Harness - Iteration 2

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.

**Independent seed and mint.** This notebook lets a human collaborator work entirely through a browser in Google Colab. No desktop IDE is required.

Upload the CVAS harness ZIP and then a component ZIP. The notebook checks archive paths and size before extraction, runs the JavaScript-first verifier, displays the report and offers the JSON report for download.

**Security:** uploaded component code is executed in this Colab runtime. Do not mount Google Drive, enter secrets, or use a runtime containing private data. Colab is not a hostile-code sandbox. Use an isolated controlled runner for untrusted submissions.


In [ ]:
from pathlib import Path
import zipfile, shutil, hashlib, json, os, subprocess, sys

WORK = Path('/content/cvas-cloud-source-harness-iteration-2')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
MAX_ARCHIVE_BYTES = 20 * 1024 * 1024
MAX_ARCHIVE_FILES = 500
MAX_SINGLE_FILE_BYTES = 10 * 1024 * 1024

def safe_extract(zip_path, destination):
    zip_path = Path(zip_path)
    if zip_path.stat().st_size > MAX_ARCHIVE_BYTES:
        raise ValueError('Archive exceeds 20 MiB limit')
    with zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        if len(infos) > MAX_ARCHIVE_FILES:
            raise ValueError('Archive contains too many entries')
        total = 0
        for info in infos:
            name = info.filename.replace('\\', '/')
            parts = Path(name).parts
            if name.startswith('/') or '..' in parts or (parts and ':' in parts[0]):
                raise ValueError(f'Unsafe archive path: {name}')
            # Reject symlinks from Unix ZIP metadata.
            mode = (info.external_attr >> 16) & 0o170000
            if mode == 0o120000:
                raise ValueError(f'Symlink not permitted: {name}')
            if info.file_size > MAX_SINGLE_FILE_BYTES:
                raise ValueError(f'File exceeds 10 MiB limit: {name}')
            if info.flag_bits & 0x1:
                raise ValueError(f'Encrypted ZIP entries are not supported: {name}')
            total += info.file_size
            if total > MAX_ARCHIVE_BYTES:
                raise ValueError('Expanded archive exceeds 20 MiB limit')
        zf.extractall(destination)

print('Runtime ready:', sys.version)


## 1. Upload the harness archive

In [ ]:
from google.colab import files
uploaded_harness = files.upload()
harness_zips = [name for name in uploaded_harness if name.lower().endswith('.zip')]
if len(harness_zips) != 1:
    raise RuntimeError('Upload exactly one CVAS harness ZIP')
HARNESS_DIR = WORK / 'harness'
HARNESS_DIR.mkdir()
safe_extract(harness_zips[0], HARNESS_DIR)
verifiers = list(HARNESS_DIR.rglob('verify.mjs'))
if len(verifiers) != 1:
    raise RuntimeError('Harness archive must contain exactly one pipeline/verify.mjs')
VERIFY = verifiers[0]
print('JavaScript verifier:', VERIFY)
print('Harness ZIP SHA256:', hashlib.sha256(Path(harness_zips[0]).read_bytes()).hexdigest())


In [ ]:
import shutil, subprocess
if shutil.which('node') is None:
    print('Installing Node.js runtime for JavaScript components...')
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y nodejs'],check=True)
print('Node:', subprocess.run(['node','--version'],capture_output=True,text=True,check=True).stdout.strip() if shutil.which('node') else 'not installed')


## 2. Upload the component ZIP

In [ ]:
uploaded_component = files.upload()
component_zips = [name for name in uploaded_component if name.lower().endswith('.zip')]
if len(component_zips) != 1:
    raise RuntimeError('Upload exactly one component ZIP')
COMPONENT_DIR = WORK / 'component'
COMPONENT_DIR.mkdir()
safe_extract(component_zips[0], COMPONENT_DIR)
manifests = list(COMPONENT_DIR.rglob('manifest.json'))
if len(manifests) != 1:
    raise RuntimeError('Component archive must contain exactly one manifest.json')
COMPONENT = manifests[0].parent
print('Component:', COMPONENT)
print('Component ZIP SHA256:', hashlib.sha256(Path(component_zips[0]).read_bytes()).hexdigest())


## 3. Run verification

In [ ]:
REPORT = WORK / 'verification-report.json'
result = subprocess.run(
    ['node', str(VERIFY), str(COMPONENT), '--json-out', str(REPORT)],
    capture_output=True, text=True, timeout=180
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
if REPORT.exists():
    report = json.loads(REPORT.read_text())
    print('FINAL OUTCOME:', report['outcome'])
    print('HUMAN REVIEW:', report.get('review', {}))
else:
    raise RuntimeError('Verifier did not produce a report')
if result.returncode != 0:
    print('This contribution is not fully accepted by the generic harness.')


## 4. Download the evidence report

`INCOMPLETE` means one or more required project-specific or human checks remain. It is not a pass. Class B requires independent technical review. Class C requires complete human examination and two independent approvals in the upstream repository's protected-branch policy.

This report does not certify safety, correctness or release readiness.

In [ ]:
from google.colab import files
files.download(str(REPORT))
